# NSE Index Futures — Architecture Comparison (transfer claim, E3)

Trains all four architectures (**DeepLOB, MLPLOB, TLOB, MambaLOB**) from scratch on **NSE NIFTY /
BANKNIFTY index-futures** LOB, using the continuous front-month series (May→June, stitched across the
expiry roll). Event-time, **Scheme A** labels (alpha=1e-5, calibrated for index-futures price scale).

This is the **transfer claim**: the comparison is *between architectures on Indian data* — not against
published FI-2010 numbers (different market/data). Per LOBCAST/TLOB, expect F1 near baseline and decaying
with horizon; that is a finding. Every result is reported **against majority / always-stationary / random
baselines** so "barely above chance" can't masquerade as success.

> Needs a **GPU** (Mamba) and **AWS credentials** (to pull the Dhan parquet from S3). Reuses the repo's
> `load_nse` (front-month stitching + per-day lazy windowing) / `build_model` / `train`.


## 1. Runtime check

In [ ]:
import torch, platform
print("Python:", platform.python_version(), "| Torch:", torch.__version__, "| CUDA:", torch.version.cuda)
assert torch.cuda.is_available(), "Enable a GPU runtime."
print("GPU:", torch.cuda.get_device_name(0))

## 2. Get the project code (reads the `GH_PAT` Colab secret)

In [ ]:
import sys, subprocess, pathlib
REPO_URL = "https://github.com/rajjoseph48/nse-lob-capstone.git"
REPO_DIR = "nse-lob-capstone"
GITHUB_TOKEN = ""
try:
    from google.colab import userdata
    GITHUB_TOKEN = GITHUB_TOKEN or (userdata.get("GH_PAT") or "")
except Exception:
    pass

def add_modeling(base):
    base = pathlib.Path(base)
    for cand in (base / "modeling", base):
        if (cand / "models.py").exists():
            sys.path.insert(0, str(cand.resolve()))
            return str(cand.resolve())
    return None

path = None
for c in (".", REPO_DIR, "/content/" + REPO_DIR):
    path = add_modeling(c)
    if path:
        break
if not path:
    url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@") if GITHUB_TOKEN else REPO_URL
    subprocess.run(["git", "clone", "--depth", "1", url], check=True)
    path = add_modeling(REPO_DIR)
print("modeling/ on sys.path:", path)

## 3. Install deps (Mamba kernel + boto3)
`mamba-ssm` compiles CUDA extensions (a few minutes). `boto3` pulls the data from S3.

In [ ]:
import subprocess, sys
def pipq(*p): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *p], check=False)
pipq("boto3")
pipq("causal-conv1d")
pipq("mamba-ssm")
print("install step done")

## 4. Download NSE Dhan data from S3

Needs AWS credentials. Store them as **Colab Secrets** `AWS_ACCESS_KEY_ID` and `AWS_SECRET_ACCESS_KEY`
(same way as `GH_PAT`), or paste below. Bucket `lob-capstone-data`, region **ap-south-2**. Downloads the
daily `lob_dhan_*.parquet` (~660 MB, skips empty roll-week files).

In [ ]:
import os, boto3, pathlib

def _secret(name):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    return os.environ.get(name, "")

AWS_ACCESS_KEY_ID = ""       # or leave blank to use the Colab secret of the same name
AWS_SECRET_ACCESS_KEY = ""
os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID or _secret("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY or _secret("AWS_SECRET_ACCESS_KEY")

BUCKET, PREFIX, REGION = "lob-capstone-data", "lob-data/dhan/", "ap-south-2"
DATA_DIR_NSE = "nse_data/dhan"
pathlib.Path(DATA_DIR_NSE).mkdir(parents=True, exist_ok=True)

s3 = boto3.client("s3", region_name=REGION)
objs = s3.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX).get("Contents", [])
n = 0
for o in objs:
    if o["Size"] == 0:
        continue                      # skip empty roll-week files
    dst = os.path.join(DATA_DIR_NSE, o["Key"].split("/")[-1])
    if not os.path.exists(dst):
        s3.download_file(BUCKET, o["Key"], dst)
    n += 1
print(f"{n} parquet files in {DATA_DIR_NSE}:", sorted(os.listdir(DATA_DIR_NSE))[:3], "...")

## 5. Metrics + naive baselines

In [ ]:
import numpy as np, torch
from sklearn.metrics import (accuracy_score, f1_score, precision_recall_fscore_support,
                             matthews_corrcoef, confusion_matrix)
from fi2010_dataset import make_loader
from train import DEVICE

CLASS_NAMES = ["Down", "Stat", "Up"]

def collect_preds(model, ds, batch_size=256):
    model = model.to(DEVICE).eval()
    loader = make_loader(ds, batch_size=batch_size, shuffle=False)
    preds, labels = [], []
    with torch.no_grad():
        for X, y in loader:
            preds.append(model(X.to(DEVICE)).argmax(1).cpu().numpy())
            labels.append(y.numpy())
    return np.concatenate(preds), np.concatenate(labels)

def full_metrics(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1, 2], zero_division=0)
    return {"accuracy": float(accuracy_score(y_true, y_pred)),
            "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
            "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
            "mcc": float(matthews_corrcoef(y_true, y_pred)),
            "per_class": {CLASS_NAMES[i]: {"precision": float(p[i]), "recall": float(r[i]),
                                           "f1": float(f[i])} for i in range(3)},
            "confusion": confusion_matrix(y_true, y_pred, labels=[0, 1, 2]).tolist()}

def naive_baselines(y_train, y_test):
    """Weighted-F1 of majority-class / always-stationary / class-freq-random predictors."""
    maj = int(np.bincount(y_train, minlength=3).argmax())
    wf1 = lambda yp: f1_score(y_test, yp, average="weighted", zero_division=0)
    rng = np.random.default_rng(0)
    probs = np.bincount(y_train, minlength=3) / len(y_train)
    return {"baseline_majority_wf1": round(wf1(np.full_like(y_test, maj)), 4),
            "baseline_stat_wf1": round(wf1(np.full_like(y_test, 1)), 4),
            "baseline_random_wf1": round(wf1(rng.choice(3, size=len(y_test), p=probs)), 4)}

## 6. NSE experiment runner
Trains one (model, symbol, horizon) on the stitched front-month series; records test metrics, the naive
baselines, and class balance. NSE overfits fast, so `patience=3`. Resumable CSV.

In [ ]:
import time, json, pathlib, pandas as pd
from nse_dataset import load_nse
from models import build_model
from train import train, save_checkpoint

RESULTS_DIR = pathlib.Path("results"); RESULTS_DIR.mkdir(exist_ok=True)
CKPT_DIR = pathlib.Path("checkpoints/nse"); CKPT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = RESULTS_DIR / "nse_matrix.csv"
MODEL_LR = {"deeplob": 1e-3, "mlplob": 1e-3, "tlob": 1e-4, "mambalob": 3e-4}

def run_nse(model_name, symbol, horizon, label_scheme="A", seed=0,
            epochs=20, patience=3, batch_size=128, seq_len=100, save_to_csv=True):
    from stats import set_seed
    set_seed(seed)
    sch = "" if label_scheme.upper() == "A" else f"_{label_scheme.upper()}"
    tag = f"{model_name}_{symbol}_h{horizon}{sch}_s{seed}"
    lr = MODEL_LR.get(model_name, 1e-3)
    print("=" * 72); print(" ", tag, f"(lr={lr}, scheme={label_scheme})"); print("=" * 72)
    tr, vl, te = load_nse(symbol=symbol, horizon=horizon, seq_len=seq_len,
                          alpha=1e-5, label_scheme=label_scheme, data_dir=DATA_DIR_NSE)
    model = build_model(model_name, seq_len=seq_len, n_features=40)
    n_params = sum(p.numel() for p in model.parameters())
    t0 = time.time()
    hist = train(model, tr, vl, epochs=epochs, patience=patience, batch_size=batch_size, lr=lr, verbose=True)
    elapsed = time.time() - t0
    y_pred, y_true = collect_preds(model, te)
    mt = full_metrics(y_true, y_pred)
    bl = naive_baselines(tr.y.numpy(), y_true)
    row = {"model": model_name, "symbol": symbol, "horizon": horizon,
           "label_scheme": label_scheme.upper(), "seed": seed, "n_params": n_params,
           "best_epoch": hist["best_epoch"], "epochs_run": len(hist["val_f1"]),
           "best_val_f1": round(max(hist["val_f1"]), 4),
           "test_accuracy": round(mt["accuracy"], 4), "test_macro_f1": round(mt["macro_f1"], 4),
           "test_weighted_f1": round(mt["weighted_f1"], 4), "test_mcc": round(mt["mcc"], 4),
           **bl, "train_time_s": round(elapsed, 1)}
    if save_to_csv:
        ckpt = CKPT_DIR / f"{tag}.pt"; save_checkpoint(model, str(ckpt)); row["checkpoint"] = str(ckpt)
        df = pd.read_csv(RESULTS_CSV) if RESULTS_CSV.exists() else pd.DataFrame()
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
        df.to_csv(RESULTS_CSV, index=False)
    print(f"  -> macro_f1={mt['macro_f1']:.4f} weighted_f1={mt['weighted_f1']:.4f} "
          f"acc={mt['accuracy']:.4f} | baselines maj/stat/rand="
          f"{bl['baseline_majority_wf1']}/{bl['baseline_stat_wf1']}/{bl['baseline_random_wf1']} ({elapsed:.0f}s)")
    return row, mt, (y_true, y_pred)

## 7. Run the matrix

4 models × {NIFTY, BANKNIFTY} × horizons {10, 20, 50, 100} = 32 runs (Scheme A). Resumable — completed
`(model, symbol, horizon)` rows are skipped. Trim `MODELS` / `HORIZONS` to save time; it's deliberately
ordered cheap→expensive (deeplob, mlplob, tlob, mambalob).

In [ ]:
MODELS = ["deeplob", "mlplob", "tlob", "mambalob"]
SYMBOLS = ["NIFTY", "BANKNIFTY"]
HORIZONS = [10, 20, 50, 100]
EPOCHS = 20

def _done_keys():
    if not RESULTS_CSV.exists():
        return set()
    d = pd.read_csv(RESULTS_CSV)
    for col, default in [("label_scheme", "A"), ("seed", 0)]:
        if col not in d.columns:
            d[col] = default
    return {(r.model, r.symbol, int(r.horizon), r.label_scheme, int(r.seed)) for r in d.itertuples()}

done = _done_keys()
for mdl in MODELS:
    for sym in SYMBOLS:
        for h in HORIZONS:
            if (mdl, sym, h, "A", 0) in done:
                print(f"skip {mdl} {sym} h{h} (in CSV)"); continue
            run_nse(mdl, sym, h, label_scheme="A", seed=0, epochs=EPOCHS)

pd.read_csv(RESULTS_CSV)

## 7b. Scheme B — spread-relative labels (E4)

Threshold θ = mean(spread/mid) over train (≈ transaction cost). **At short horizons Scheme B is
degenerate** — the spread (~2.5 bps for NIFTY) dwarfs a 10-event return, so ~99% of windows are
Stationary; that itself is a transaction-cost-relative-predictability finding. So Scheme B is run at
**longer horizons** where cumulative moves can exceed the spread. The loader prints the class balance;
if Stationary > 95% the task is degenerate and we note it rather than chase it.

In [ ]:
HORIZONS_B = [50, 100, 200]   # short horizons are ~all-Stationary under Scheme B
for mdl in MODELS:
    for sym in SYMBOLS:
        for h in HORIZONS_B:
            if (mdl, sym, h, "B", 0) in _done_keys():
                print(f"skip {mdl} {sym} h{h} B (in CSV)"); continue
            run_nse(mdl, sym, h, label_scheme="B", seed=0, epochs=EPOCHS)
pd.read_csv(RESULTS_CSV).query("label_scheme == 'B'")

## 7c. Multi-seed + bootstrap CIs on the headline (NIFTY, Scheme A)

Single runs hide variance. Here each headline config is trained over **3 seeds** → report mean ± std of
macro-F1, plus a **bootstrap 95% CI** on the test set (from seed 0's predictions). Writes a separate
`nse_seeds.csv` so it doesn't pollute the main matrix.

In [ ]:
from stats import seed_summary, bootstrap_ci
SEEDS = [0, 1, 2]
HEADLINE_SYMBOL, HEADLINE_HORIZON = "NIFTY", 10
SEEDS_CSV = RESULTS_DIR / "nse_seeds.csv"

seed_rows = []
for mdl in MODELS:
    macro_by_seed, first_preds = [], None
    for s in SEEDS:
        _, mt, (yt, yp) = run_nse(mdl, HEADLINE_SYMBOL, HEADLINE_HORIZON,
                                  label_scheme="A", seed=s, epochs=EPOCHS, save_to_csv=False)
        macro_by_seed.append(mt["macro_f1"])
        if first_preds is None:
            first_preds = (yt, yp)
    summ = seed_summary(macro_by_seed)
    ci = bootstrap_ci(*first_preds, average="macro", n_boot=1000)
    seed_rows.append({"model": mdl, "symbol": HEADLINE_SYMBOL, "horizon": HEADLINE_HORIZON,
                      "macro_f1_mean": round(summ["mean"], 4), "macro_f1_std": round(summ["std"], 4),
                      "seeds": summ["values"], "ci_low": round(ci["ci_low"], 4),
                      "ci_high": round(ci["ci_high"], 4)})
    print(f"{mdl}: macro-F1 {summ['mean']:.4f} ± {summ['std']:.4f}  "
          f"95% CI [{ci['ci_low']:.4f}, {ci['ci_high']:.4f}]")
seeds_df = pd.DataFrame(seed_rows); seeds_df.to_csv(SEEDS_CSV, index=False)
seeds_df

## 8. Results vs baselines

The honest read: how far above the best naive baseline (usually always-stationary) does each architecture
get? Small/negative gaps are the expected, reportable LOBCAST-style finding.

In [ ]:
import pandas as pd
res = pd.read_csv(RESULTS_CSV)
res["best_baseline_wf1"] = res[["baseline_majority_wf1", "baseline_stat_wf1", "baseline_random_wf1"]].max(axis=1)
res["lift_vs_baseline"] = (res["test_weighted_f1"] - res["best_baseline_wf1"]).round(4)
print("Weighted-F1 by model x horizon (NIFTY):")
print(res[res.symbol == "NIFTY"].pivot_table(index="horizon", columns="model",
      values="test_weighted_f1").round(4).to_string())
print("\nLift over best naive baseline (weighted-F1):")
print(res.pivot_table(index=["symbol", "horizon"], columns="model",
      values="lift_vs_baseline").round(4).to_string())
res.sort_values(["symbol", "model", "horizon"])

## 9. Persist results + checkpoints (S3 / Drive)

In [ ]:
import glob
S3_BUCKET, S3_PREFIX, S3_REGION = "lob-capstone-data", "reproduction/nse", "ap-south-2"
try:
    import boto3
    s3 = boto3.client("s3", region_name=S3_REGION)
    for f in glob.glob("results/*") + glob.glob("checkpoints/nse/*"):
        key = f"{S3_PREFIX}/{f}"; s3.upload_file(f, S3_BUCKET, key); print("uploaded", key)
    print(f"Done -> s3://{S3_BUCKET}/{S3_PREFIX}/")
except Exception as e:
    print("S3 upload skipped:", repr(e))
# Drive: from google.colab import drive; drive.mount("/content/drive"); import shutil
#        shutil.copytree("results", "/content/drive/MyDrive/nse_matrix/results", dirs_exist_ok=True)